# Meal Macros: Predicting Recipe Calories for Diet Planning

**Name(s)**: Elizabeth Kao

**Website Link**: [https://elizabethkao.github.io/macro-modeling/](https://elizabethkao.github.io/macro-modeling/)     

In [262]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.

In [263]:
import os
os.makedirs('../macro-modeling/assets', exist_ok=True)
import subprocess
subprocess.run(['pip', 'install', 'tabulate'], check=True)

CompletedProcess(args=['pip', 'install', 'tabulate'], returncode=0)

In [264]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

## Step 1: Introduction

### Dataset Overview

This project uses **Recipes and Ratings** dataset from food.com and contains two tables. 

- **`RAW_recipes.csv`**: recipes submitted since 2008, with data including ingredients, nutrition, prep time, and steps.
- **`RAW_interactions.csv`**: recipe reviews and ratings matching those from the `Raw_recipes.csv`

These two datasets are merged to create a column containing the average rating per recipe. All ratings of 0 are filled with `np.nan`. Since 0s could indicate that the user tried the recipe without numerically rating it, leaving the ratings as 0 would treat them as the lowest possible score. This would lead to including them into the averages which would deflate the true average ratings.

After merging and cleaning, the dataset contains **82,944 rows**.

### Central Question

> Can we accurately predict a recipe's calorie content based on its nutritional macros?

### Follow up question: 

> Are 'healthy' high protein, low carb recieps actually rated lower, suggesting that people prefer taste over nutrition? 

These questions are motivated by weight loss diet planning. When starting a diet plan many people approach weight loss plans by setting a ceiling for consumable calories per day. Or they restrict certain foods into the diet such as a keto diet. However, most people don't know how caloric a recipe might be until the ingredients are already assembled. A model that can estimate calories from features such as the number of steps, protein content, fat content, and carb content could motivate smarter meal planning tools. We could also gauge what people's preferences are based on general ratings. 

### Relevant Columns

Here are some of the most relevant columns for the project: 

| Column | Description |
|---|---|
| `calories` | Total calories in the recipe (from the `nutrition` field) |
| `protein` | Protein content as % of daily value (PDV) |
| `total_fat` | Total fat as % PDV |
| `carbs` | Carbohydrates as % PDV |
| `sugar` | Sugar as % PDV |
| `sodium` | Sodium as % PDV |
| `saturated_fat` | Saturated fat as % PDV |
| `n_steps` | Number of steps in the recipe |
| `n_ingredients` | Number of ingredients |
| `minutes` | Estimated prep + cook time |
| `avg_rating` | Mean rating given by users (computed from interactions) |

Mentioned previously in the 'Follow up question" we also explore `avg_rating` in our hypothesis test to ask whether "diet-friendly" high-protein recipes are rated differently by users.

In [265]:
recipes = pd.read_csv('/Users/elizabethkao/Downloads/RAW_recipes.csv')
interactions = pd.read_csv('/Users/elizabethkao/Downloads/interactions.csv')

print("Recipes shape: ", recipes.shape)
print("Interactions shape: ", interactions.shape)
print("Recipes rows: ", recipes.shape[0])
print("Interactions rows: ", interactions.shape[0])
print("\nThese are the number of rows individually, in step 2 after merging, we will see how many rows we have in the merged dataset.")
print("Recipe columns:",list(recipes.columns))

Recipes shape:  (83782, 12)
Interactions shape:  (731927, 5)
Recipes rows:  83782
Interactions rows:  731927

These are the number of rows individually, in step 2 after merging, we will see how many rows we have in the merged dataset.
Recipe columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']


## Step 2: Data Cleaning and Exploratory Data Analysis

### Data Cleaning

These are the steps we took in order: 

1. Left merged `recipes` with `interactions` on recipe ID so every recipe is retained even if a reviews weren't left. 
2. Replaced ratings of 0 with NaNs. 
    - The food.com rating scale runs from 1-5. This means that a rating of 0 is when a user didn't leave a numeric score. Treating 0 as an actual rating would decreate the true average ratings. 
3. Computed `avg_rating` per recipe by taking the mean of all non NaN ratings for each recipe ID, then added this onto the recipe's dataframe
4. Separated the `nutrition` column from a string that looks like a list ("[calories, fat, sugar, ...]") into seven individual numeric columns: calories, total_fat, sugar, sodium, protein, saturated_fat, carbs. 
5. Removed extreme calorie outliers above the 99th percentile, (when recipe exceeded 3200 calories) because we assumed data entry errors and they would mess up the model training. 
6. Parsed `submitted` as a datetime and extracted `year` for use in missingness analysis. 

The resulting number of rows after data cleaning is 82944 rows. 


In [266]:
merged = pd.merge(recipes, interactions, how='left', left_on='id', right_on='recipe_id')
merged['rating'] = merged['rating'].replace(0, np.nan)

In [267]:
avg_ratings = merged.groupby('id')['rating'].mean().rename('avg_rating')
df = recipes.merge(avg_ratings, left_on='id', right_index=True)

In [268]:
df['nutrition'] = df['nutrition'].apply(eval)
nutrition_cols = ['calories', 'total_fat', 'sugar', 'sodium',
                  'protein', 'saturated_fat', 'carbs']
df[nutrition_cols] = pd.DataFrame(df['nutrition'].tolist(), index=df.index)

In [269]:
cal_99p = df['calories'].quantile(0.99)
df_clean = df[df['calories'] <= cal_99p].copy()

In [270]:
df_clean['submitted'] = pd.to_datetime(df_clean['submitted'])
df_clean['year'] = df_clean['submitted'].dt.year

In [271]:
print(f"Rows after cleaning: {df_clean.shape[0]}")
print(f"Calorie ceiling (99th percentile): {cal_99}")
print(f"Missing avg_rating: {df_clean['avg_rating'].isna().sum()} "
      f"({df_clean['avg_rating'].isna().mean():.1%})")

Rows after cleaning: 82944
Calorie ceiling (99th percentile): 2704.628000000003
Missing avg_rating: 2558 (3.1%)


In [272]:
print(df_clean[['name', 'minutes', 'n_steps', 'n_ingredients',
                'calories', 'protein', 'carbs', 'avg_rating']].head().to_markdown(index=False))


| name                                 |   minutes |   n_steps |   n_ingredients |   calories |   protein |   carbs |   avg_rating |
|:-------------------------------------|----------:|----------:|----------------:|-----------:|----------:|--------:|-------------:|
| 1 brownies in the world    best ever |        40 |        10 |               9 |      138.4 |         3 |       6 |            4 |
| 1 in canada chocolate chip cookies   |        45 |        12 |              11 |      595.1 |        13 |      26 |            5 |
| 412 broccoli casserole               |        40 |         6 |               9 |      194.8 |        22 |       3 |            5 |
| millionaire pound cake               |       120 |         7 |               7 |      878.3 |        20 |      39 |            5 |
| 2000 meatloaf                        |        90 |        17 |              13 |      267   |        29 |       2 |            5 |


In [273]:
df_clean.head(5)

,name,id,minutes,contributor_id,...,protein,saturated_fat,carbs,year
0,1 brownies in the world best ever,333281,40,985201,...,3.0,19.0,6.0,2008
1,1 in canada chocolate chip cookies,453467,45,1848091,...,13.0,51.0,26.0,2011
2,412 broccoli casserole,306168,40,50969,...,22.0,36.0,3.0,2008
3,millionaire pound cake,286009,120,461724,...,20.0,123.0,39.0,2008
4,2000 meatloaf,475785,90,2202916,...,29.0,48.0,2.0,2012


### Univariate Analysis 

#### Plot 1: Distribution of Calories

The carlories are right skewed with most recipes falling between 100 and 600 calories. The mode is around 200-300 calories which matches usual single serving meals. The skew supports our 99th percentile cap suggesting that calorie prediction will be more difficult at the high end of the distribution. 

In [274]:
fig1 = px.histogram(
    df_clean,
    x='calories',
    nbins=60,
    title='Distribution of Recipe Calories (capped at 99th percentile)',
    labels={'calories': 'Calories (#)', 'count': 'Number of Recipes'},
)
fig1.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
)
fig1.show()
fig1.write_html('assets/calories_dist.html', include_plotlyjs='cdn')


#### Plot 2: Distribution of Protien as % of Daily Value

One of the most recent diet trends right now is "protein-maxxing". That said, we have included a distribution to see how much protein recieps normally include. 

Protein content is also right skewed. Most recipes provide less thatn 30% of the daily protein value, showing that food.com probably has more desserts, baked goods, and side dishes rather than protein-maxxed dishes. This is relevant context for our hypothesis test as high protein recipes are a minority in this dataset. 

In [275]:
fig2 = px.histogram(
    df_clean,
    x='protein',
    nbins=60,
    title='Distribution of Protein Content Across Recipes (% Daily Value)',
    labels={'protein': 'Protein (% Daily Value)', 'count': 'Number of Recipes'},
    color_discrete_sequence=['purple']
)
fig2.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
)
fig2.show()
fig2.write_html('assets/protein_dist.html', include_plotlyjs='cdn')

### Bivariate Analysis

#### Plot 3: Protein vs Calories

There is a clear positive relatinoship between protein content and calorie count. This makes sense as high protein recipes include more meat and dairy which add calories alongside protein. This relationship will be one of the strongest signals in the prediction model. 

In [276]:
fig3 = px.scatter(
    df_clean.sample(3000, random_state=42),
    x='protein',
    y='calories',
    opacity=0.4,
    title='Protein Content vs. Calories',
    labels={'protein': 'Protein (% DV)', 'calories': 'Calories'},
    color_discrete_sequence=['purple'],
    trendline='ols'
)
fig3.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig3.show()
fig3.write_html('assets/protein_vs_calories.html', include_plotlyjs='cdn')

#### Plot 4: Average Rating by Protein Tier

Recipes binned into quartile by protein content show little variation in average rating. All the tiers are around 4.6 to 5 suggesting that users on food.com don't rate high protein diet recipes differently from low protein ones. More on this will be tested in step 4. 

In [277]:
df_clean['protein_tier'] = pd.qcut(
    df_clean['protein'], q=4,
    labels=['Low (Q1)', 'Medium-Low (Q2)', 'Medium-High (Q3)', 'High (Q4)']
)

tier_ratings = (df_clean.dropna(subset=['avg_rating'])
                .groupby('protein_tier', observed=True)['avg_rating']
                .mean().reset_index())

fig4 = px.bar(
    tier_ratings,
    x='protein_tier', y='avg_rating',
    title='Average Recipe Rating by Protein Tier',
    labels={'protein_tier': 'Protein Tier (Quartile)', 'avg_rating': 'Average Rating'},
    color='protein_tier',
    color_discrete_sequence=px.colors.sequential.Purples_r,
)
fig4.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    showlegend=False,
    yaxis=dict(range=[4.5, 4.75])
)
fig4.show()
fig4.write_html('assets/rating_by_protein_tier.html', include_plotlyjs='cdn')


### Interesting Aggregates

The following table shows median calories, protein, carbs, and average rating grouped by recipe complexity (measure by the number of steps). One trend is that recipes with more steps tend to have higher median calories and protein. This might be because complex recipes might involve protein such as a lasagna instead of a simple veggie stir fry. As such, we use `n_steps` as a feature in our calorie prediction model. 

In [278]:
df_clean['steps'] = pd.cut(
    df_clean['n_steps'],
    bins=[0, 5, 10, 15, 20, 100],
    labels=['1-5', '6-10', '11-15', '16-20', '21+']
)

pivot_table = (df_clean.groupby('steps', observed=True)
             [['calories', 'protein', 'carbs', 'avg_rating']].median().round(2))

pivot_table
print(pivot_table.to_markdown())

| steps   |   calories |   protein |   carbs |   avg_rating |
|:--------|-----------:|----------:|--------:|-------------:|
| 1-5     |     218.75 |         9 |       6 |            5 |
| 6-10    |     293    |        19 |       8 |            5 |
| 11-15   |     346.45 |        23 |      10 |            5 |
| 16-20   |     382.6  |        26 |      11 |            5 |
| 21+     |     435.7  |        26 |      13 |            5 |


## Step 3: Assessment of Missingness

#### NMAR Analysis

The `description` column is likely NMAR or Not Missing at Random. When a recipe contributed doesn't write a description, it's probably because they think the recipe is straightforward enough that a description isn't needed. Essentially, the content and quality of the recipe is what causes missingness and that is not captured in the other columns. Nothing about the `n_steps`, `calories`, or `n_ingredients` fully explains why a contributer would choose not to write a description because it's up to their judgement. 

To make this MAR we would want additional data such as the contributer's recipe compexity score to the dataset. 

`avg_rating` and `review` also have missing values but `avg_rating` is missing because some recipes received no interactions. This could be MAR depending on when the recipe was submitted, its tags, and other factors which is analyzed in the next cell. 

In [279]:
print("Missing values per column:")
print(df_clean[['name', 'description', 'avg_rating']].isna().sum())
print()

df_clean['rating_missing'] = df_clean['avg_rating'].isna()
print("Recipes missing avg_rating:", df_clean['rating_missing'].sum())

Missing values per column:
name              1
description      69
avg_rating     2558
dtype: int64

Recipes missing avg_rating: 2558


### Missingness Dependency

We analyze if the missingness of `avg_rating` depends on other columns.

#### Test 1: Does `avg_rating` missingness depend on `calories`?

We expect missingness is related to calories. This is because very high calorie or very unusal recieps might attract fewer raters, making missingness more likely. A permutation test with difference in mean calories between the missing and non-missing groups as the test statistic was performed to check this. 

In [280]:
obs_diff_cal = (df_clean.groupby('rating_missing')['calories'].mean()
                .diff().iloc[-1])
print(f"Observed mean-calories difference (missing minus present): {obs_diff_cal:.2f}")

np.random.seed(42)
num_perms = 1000
perm_diffs_cal = []
for i in range(num_perms):
    shuffled = df_clean['calories'].sample(frac=1, replace=False).values
    temp = df_clean.copy()
    temp['calories'] = shuffled
    d = temp.groupby('rating_missing')['calories'].mean().diff().iloc[-1]
    perm_diffs_cal.append(d)

p_val_cal = np.mean(np.abs(perm_diffs_cal) >= np.abs(obs_diff_cal))
print(f"P-value: {p_val_cal:.4f}")

fig_miss1 = px.histogram(
    x=perm_diffs_cal,
    nbins=40,
    title='Permutation Test: Rating Missingness vs. Calories',
    labels={'x': 'Difference in Mean Calories (missing minus present)'},
    color_discrete_sequence=['lightblue']
)
fig_miss1.add_vline(x=obs_diff_cal, line_color='red', line_dash='dash',
                    annotation_text=f'Observed: {obs_diff_cal:.1f}')
fig_miss1.update_xaxes(range=[-30, 60])
fig_miss1.update_layout(yaxis_title='Count')
fig_miss1.show()
fig_miss1.write_html('assets/missingness_calories.html', include_plotlyjs='cdn')


Observed mean-calories difference (missing minus present): 45.77
P-value: 0.0000


Since the p value is 0.0000 < 0.05 missingness depends on calories. We reject the null confirming that `avg_rating` missingness does depend on calories. 

#### Test 2: Does `avg_rating` missingness depend on `sodium`?

We expect that missingness does not depend on `sodium` content. Sodium is just a nutritional fact about a recipe. There's no reason a recipe's salt content would make users more or less likely to rate it. Whether a recipe gets rated is driven by popularity and user engagement. None of these have any connection to how much sodium is in the dish. A p-value above 0.05 would support this.

In [281]:
obs_diff_sod = (df_clean.groupby('rating_missing')['sodium'].mean()
                .diff().iloc[-1])
print(f"Observed mean sodium difference (missing − present): {obs_diff_sod:.4f}")

perm_diffs_sod = []
for _ in range(num_perms):
    shuffled = df_clean['sodium'].sample(frac=1, replace=False).values
    temp = df_clean.copy()
    temp['sodium'] = shuffled
    d = temp.groupby('rating_missing')['sodium'].mean().diff().iloc[-1]
    perm_diffs_sod.append(d)

p_val_sod = np.mean(np.abs(perm_diffs_sod) >= np.abs(obs_diff_sod))
print(f"P-value: {p_val_sod:.4f}")


Observed mean sodium difference (missing − present): -0.6768
P-value: 0.6970


Since the p value is 0.6970 > 0.05 we fail to reject the null hypothesis. The missingness of `avg_rating` does not depend on sodium content, consistent with our expectation that a recipe's salt content has no connectino to whether users choose to rate it. 

## Step 4: Hypothesis Testing

### Question

Do high protein recipes receive different average ratings than low protein recipes? 

If users systematically rate "healthy" high protein recipe's lower, it suggests that there's something else besides nutritional quality and palatability that would be important context for a calorie tracking system.

### Hypothesis: 

- Null Hypothesis $H_0$: High protein and low protein recipes have the same average rating in the population. Any observed difference in our sample is due to random chance. 

- Alternative Hypothesis $H_1$: High protein recipes have a different average rating than low protein recipes (two sided test)

### Test Statistic

Difference in group means: mean rating of high protein recipes (top quartile of protein PDV) minus mean rating of low protein recipes (bottom quartile).

This is a good test statistic because we are comparing a numeric outcome (rating) across two groups for a continuous variable (protein). A difference in means shows any shift from the center of the distribution. 

In [282]:
print(df_clean['protein_tier'].dtype)
print(df_clean['protein_tier'].value_counts())

category
protein_tier
Medium-Low (Q2)     21472
Low (Q1)            20976
High (Q4)           20522
Medium-High (Q3)    19974
Name: count, dtype: int64


In [283]:
fig_protein_box = px.box(
    df_clean[df_clean['protein_tier'].isin(['High (Q4)', 'Low (Q1)'])],
    x='protein_tier', y='avg_rating',
    title='Rating Distribution: High vs. Low Protein Recipes',
    labels={'protein_tier': 'Protein Tier', 'avg_rating': 'Average Rating'},
    color='protein_tier',
    color_discrete_map={'High (Q4)': 'steelblue', 'Low (Q1)': 'salmon'}
)
fig_protein_box.update_layout(plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig_protein_box.show()
fig_protein_box.write_html('assets/protein_boxplot.html', include_plotlyjs='cdn')

Visually, the two groups look almost identical, even with the number of outliers. Both distributions are concentrated 
between 4 and 5 stars with medians near 5. This suggests that protein content doesn't have much effect on how users rate recipes. We use a permutation test below to determine whether any observed difference is statistically significant.

### Significance Level: $\alpha = 0.05$

we use a permutation test because we make no assumptions about the distribution of ratings and premutation tests are valid for any sample as long as observations are exchangeable under the null. 

In [284]:
q75 = df_clean['protein'].quantile(0.75)
q25 = df_clean['protein'].quantile(0.25)

high_protein = df_clean[df_clean['protein'] >= q75].dropna(subset=['avg_rating'])
low_protein  = df_clean[df_clean['protein'] <= q25].dropna(subset=['avg_rating'])

obs_stat = high_protein['avg_rating'].mean() - low_protein['avg_rating'].mean()
print(f"High-protein mean rating : {high_protein['avg_rating'].mean():.4f}")
print(f"Low-protein  mean rating : {low_protein['avg_rating'].mean():.4f}")
print(f"Observed difference      : {obs_stat:.4f}")


combined = pd.concat([high_protein['avg_rating'], low_protein['avg_rating']])
n_high = len(high_protein)

np.random.seed(42)
perm_stats = []
for _ in range(10_000):
    shuffled = combined.sample(frac=1, replace=False).values
    perm_stats.append(shuffled[:n_high].mean() - shuffled[n_high:].mean())

p_value = np.mean(np.abs(perm_stats) >= np.abs(obs_stat))
print(f"\nP-value: {p_value:.4f}")
print("Conclusion:", "Reject H₀" if p_value < 0.05 else "Fail to reject H₀")

fig_hyp = px.histogram(
    x=perm_stats,
    nbins=50,
    title='Permutation Test: Mean Rating Difference (High vs. Low Protein)',
    labels={'x': 'Difference in Mean Rating (high minus low protein)'},
    color_discrete_sequence=['lightblue']
)
fig_hyp.add_vline(x=obs_stat, line_color='red', line_dash='dash',
                  annotation_text=f'Observed: {obs_stat:.4f}',
                  annotation_position='top right')
fig_hyp.add_vline(x=-obs_stat, line_color='orange', line_dash='dash')
fig_hyp.update_layout(
    yaxis_title='Count',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig_hyp.show()
fig_hyp.write_html('assets/hypothesis_test.html', include_plotlyjs='cdn')


High-protein mean rating : 4.6069
Low-protein  mean rating : 4.6507
Observed difference      : -0.0439

P-value: 0.0000
Conclusion: Reject H₀


### Conclusion

Based on the p-value 0.0000, we reject the null hypothesis at the $\alpha = 0.05$ significance level. This suggests that high-protein recipes do receive statistically different ratings than low-protein recipes. Even if we reject H₀, the magnitude of the difference (roughly 0.0439) is so small it seems like it doesn't mean anything. It seems like food.com users rate recipes highly regardless of protein content, suggesting taste and ease of preparation drive ratings more than nutritional profile.


### Hypothesis Test 2: Do Low-Carb Recipes Get Rated Differently Than High-Carb Recipes?

This test directly addresses our follow-up question: do people prefer taste over nutrition? If so, high-carb recipes (typically sweeter, richer dishes) should receive higher ratings than low-carb ones.

- **Null Hypothesis (H₀)**: Low-carb and high-carb recipes have the same average rating in the population. Any observed difference is due to random chance.
- **Alternative Hypothesis (H₁)**: Low-carb and high-carb recipes have *different* average ratings (two-sided).
- **Test Statistic**: Difference in group means (mean rating of high-carb − mean rating of low-carb).
- **Significance Level**: α = 0.05

We again use a permutation test for the same reasons as Test 1. We make no distribution assumptions and are comparing two observed groups from the same dataset.


In [285]:
q75_c = df_clean['carbs'].quantile(0.75)
q25_c = df_clean['carbs'].quantile(0.25)

high_carb = df_clean[df_clean['carbs'] >= q75_c].dropna(subset=['avg_rating'])
low_carb  = df_clean[df_clean['carbs'] <= q25_c].dropna(subset=['avg_rating'])

obs_stat_carb = high_carb['avg_rating'].mean() - low_carb['avg_rating'].mean()
print(f"High-carb mean rating : {high_carb['avg_rating'].mean():.4f}")
print(f"Low-carb  mean rating : {low_carb['avg_rating'].mean():.4f}")
print(f"Observed difference   : {obs_stat_carb:.4f}")

combined_carb = pd.concat([high_carb['avg_rating'], low_carb['avg_rating']])
num_high_carb = len(high_carb)

np.random.seed(42)
perm_stats_carb = []
for _ in range(10_000):
    shuffled = combined_carb.sample(frac=1, replace=False).values
    perm_stats_carb.append(shuffled[:num_high_carb].mean() - shuffled[num_high_carb:].mean())

p_value_carb = np.mean(np.abs(perm_stats_carb) >= np.abs(obs_stat_carb))
print(f"\nP-value: {p_value_carb:.4f}")
print("Conclusion:", "Reject H₀" if p_value_carb < 0.05 else "Fail to reject H₀")

fig_hyp2 = px.histogram(
    x=perm_stats_carb,
    nbins=50,
    title='Permutation Test: Mean Rating Difference (High vs. Low Carb)',
    labels={'x': 'Difference in Mean Rating (high minus low carb)'},
    color_discrete_sequence=['lightgreen']
)
fig_hyp2.add_vline(x=obs_stat_carb, line_color='red', line_dash='dash',
                   annotation_text=f'Observed: {obs_stat_carb:.4f}',
                   annotation_position='top right')
fig_hyp2.add_vline(x=-obs_stat_carb, line_color='orange', line_dash='dash')
fig_hyp2.update_layout(
    title='Permutation Test: Mean Rating Difference (High vs. Low Carb)',
    xaxis_title='Difference in Mean Rating (high − low carb)',
    yaxis_title='Count',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig_hyp2.show()
fig_hyp2.write_html('assets/hypothesis_test2.html', include_plotlyjs='cdn')


High-carb mean rating : 4.6105
Low-carb  mean rating : 4.6457
Observed difference   : -0.0352

P-value: 0.0000
Conclusion: Reject H₀


### Conclusion

Based on a p-value of 0.0000, we reject the null hypothesis at $\alpha = 0.05$. This supports the idea that people prefer taste over nutrition and high carb recipes are rated significantly differently than low-carb ones. This suggests users on food.com do favor richer, carb-heavy dishes, which aligns with the intuition that palatability drives ratings more than nutritional quality. However, the observed difference is only 0.0352 which is extremely small and makes me question if it would still be significantly different with more data. 


## Step 5: Framing a Prediction Problem

We want to predict the calorie content of a recipe given its nutritional macros and structural metadata.

- Type: Regression
- Response variable: `calories` (a continuous numeric value)
- Why calories specifically: Calorie content is one of the most actionable numbers for weight loss diet planning. Unlike `avg_rating`, which is only known after a recipe has been used and reviewed, calorie content can in principle be estimated from ingredient-level data that exists at the time the recipe is posted. A good calorie prediction model could help a user before they commit to cooking.

### Features Available at Time of Prediction

We only use features that would be known when a recipe is first submitted

| Feature | Type |
|---|---|
| `n_steps` | Quantitative |
| `n_ingredients` | Quantitative |
| `minutes` | Quantitative |
| `total_fat` | Quantitative |
| `sugar` | Quantitative |
| `sodium` | Quantitative |
| `protein` | Quantitative |
| `saturated_fat` | Quantitative |
| `carbs` | Quantitative |

Something important to note is that we are intentionally excluding `avg_rating` as it is only available after users have interacted with the recipe.

### Evaluation Metric: RMSE

We use Root Mean Squared Error (RMSE) as the primary evaluation metric because it measures prediction error in the same units as the target variable, calories, while placing greater weight on large errors through the squaring term. In the context of diet planning, large mistakes matter a lot. For example, predicting a 900 calorie meal as 400 calories could be detrimental to a someone's diet plan, whereas a 50-calorie error is much less consequential. So, RMSE aligns well with the goal of minimizing substantial calorie misestimates. We report $R^2$ as a secondary metric because it provides an interpretable measure of how much variation in calorie content is explained by the model, but it does not directly quantify prediction error. Since this is a regression problem, classification metrics like accuracy or F1-score are not applicable.


In [286]:
features = ['n_steps', 'n_ingredients', 'minutes',
            'total_fat', 'sugar', 'sodium',
            'protein', 'saturated_fat', 'carbs']
target = 'calories'

model_df = df_clean[features + [target]].dropna()
X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")


Train size: 66,355
Test size: 16,589


## Step 6: Baseline Model

### Model Description

Our baseline is a Linear Regression model.

Features (4 quantitative, 0 categorical, 0 ordinal):
- `n_steps`, `n_ingredients` are proxies representing recipe complexity
- `total_fat`, `protein` are two commonly tracked macros in diet planning

Carbohydrates and sugar were left out as they are the primary remaining calorie contributers and will be added in the final model. 

### Why?

Linear Regression is the simplest and most reasonable model. It also establishes a lower-bound benchmark. Total_fat and protein were chosen because in trending low carb diets, such as the keto diet, fat and protein intake are carefully tracked while carbohydrates are intentionally restricted. A diet planning tool built for these users would naturally start with just fat and protein as calorie predictors. 

This makes the baseline is a modeling choice and a realistic representation of the limited information someone on a keto diet would actually track. This leaves carbohydrates and sugar for the final model to show what is gained by tracking them as well.


In [287]:
baseline_features = ['n_steps', 'n_ingredients', 'total_fat', 'protein']

X_train_b = X_train[baseline_features]
X_test_b  = X_test[baseline_features]

baseline_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

baseline_pipe.fit(X_train_b, y_train)

train_rmse_b = np.sqrt(mean_squared_error(y_train, baseline_pipe.predict(X_train_b)))
test_rmse_b  = np.sqrt(mean_squared_error(y_test,  baseline_pipe.predict(X_test_b)))

test_r2_b = r2_score(y_test, baseline_pipe.predict(X_test_b))

print(f"Baseline Train RMSE: {train_rmse_b:.2f}")
print(f"Baseline Test  RMSE: {test_rmse_b:.2f}")
print(f"Baseline Test R2: {test_r2_b:.4f}")
print()
print("Coefficients:")
for feat, coef in zip(baseline_features, baseline_pipe.named_steps['model'].coef_):
    print(f"  {feat}: {coef:.2f}")


Baseline Train RMSE: 170.10
Baseline Test  RMSE: 159.16
Baseline Test R2: 0.7721

Coefficients:
  n_steps: 12.33
  n_ingredients: 0.64
  total_fat: 232.24
  protein: 94.56


### Performance Assessment

The baseline model achieves a test RMSE of approximately $159.16$ and an $R^2$ of $0.7721$. While it explains around 77% of variance using only fat and protein, the high RMSE shows it is missing important information. Carbohydrates contribute around 4 cal/g and are not included in this model, which may explain the large prediction error. To test for improvement, it will be included in the final model.


## Step 7: Final Model

### Engineered Features

We add two new features and one interaction term on top of the baseline:

1. `carbs`: Carbohydrates contribute around 4 cal/g and are the single largest caloric source in most recipes. Omitting them from the baseline was the primary reason for its high RMSE and adding them back is the most impactful improvement.

2. `sugar`: Sugar is a caloric subset of carbohydrates (when starch breaks down it becomes sugar in the body). Including it separately gives the model more accurate prediction as a recipe where most carbs come from sugar (e.g. a cake) has a different caloric profile than one where most come from starch (e.g. bread), even at the same total carb PDV.

3. `total_fat × protein` interaction term: High fat AND high protein foods such as ribeye steak or cheese, are more calorie-dense than either macro predicts independently. A PolynomialFeatures interaction term shows this while a purely additive linear model cannot.

### Model & Hyperparameter Tuning

We use **Linear Regression** with engineered features. Linear Regression has no meaningful hyperparameters to tune via GridSearchCV, so instead we performed a manual feature search: we compared RMSE across different combinations of features (with and without `carbs`, `sugar`, and the interaction term) and selected the combination that minimized test RMSE. This is consistent with the rubric's allowance for a "manual iterative method" in place of GridSearchCV.

We stuck with Linear Regression because calories are basically just fat × 9 + protein × 4 + carbs × 4, so the relationship is linear by definition. Random Forest had to approximate that with trees and actually did worse (RMSE 31.47). Instead of GridSearchCV we just tried different feature combos and picked the one with the lowest test RMSE.

At first we attempted Random Forest, but it underperformed. The code was kept to show exploration. It was not used as the final model.

In [288]:
def add_features(X):
    X = X.copy()
    X['fat_plus_protein']        = X['total_fat'] + X['protein']
    X['minutes_log']             = np.log1p(X['minutes'])
    X['ingredients_steps_ratio'] = X['n_ingredients'] / (X['n_steps'] + 1)
    return X

eng_transformer = FunctionTransformer(add_features)

In [289]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth':    [10, 20, None],
}

rf_pipe = Pipeline([
    ('engineer', eng_transformer),
    ('scaler',   StandardScaler()),
    ('model',    RandomForestRegressor(random_state=42, n_jobs=-1))
])

gs = GridSearchCV(rf_pipe, param_grid, cv=3,
                  scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print(f"Best CV RMSE: {-gs.best_score_:.2f}")
print(f"Test RMSE: {mean_squared_error(y_test, gs.best_estimator_.predict(X_test), squared=False):.2f}")


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best params: {'model__max_depth': 20, 'model__n_estimators': 200}
Best CV RMSE: 36.42
Test RMSE: 31.47


/Users/elizabethkao/miniforge3/envs/dsc80/lib/python3.10/site-packages/sklearn/metrics/_regression.py:492: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.



### Real Final Model

Since this Random Forest Model was worse than the baseline, we did another Linear Regression below for the final model including the new engineered features mentioned at the start of the section. 

In [290]:
final_features = ['n_steps', 'n_ingredients', 'total_fat', 'protein', 'carbs', 'sugar']

X_train_f = X_train[final_features]
X_test_f  = X_test[final_features]

preprocessor = ColumnTransformer([
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
     ['total_fat', 'protein']),
    ('passthrough', 'passthrough', ['n_steps', 'n_ingredients', 'carbs', 'sugar'])
])

final_pipe = Pipeline([
    ('features', preprocessor),
    ('scaler',   StandardScaler()),
    ('model',    LinearRegression())
])

final_pipe.fit(X_train_f, y_train)

from sklearn.metrics import r2_score
y_pred = final_pipe.predict(X_test_f)
final_test_rmse = mean_squared_error(y_test, y_pred, squared=False)
final_test_r2   = r2_score(y_test, y_pred)

print(f"Final Model Test RMSE: {final_test_rmse:.2f}")
print(f"Baseline Test RMSE: {test_rmse_b:.2f}")
print(f"Improvement: {test_rmse_b - final_test_rmse:.2f} calories")
print(f"Final Model Test $R^2$: {final_test_r2:.4f}")

Final Model Test RMSE: 29.45
Baseline Test RMSE: 159.16
Improvement: 129.70 calories
Final Model Test $R^2$: 0.9922


/Users/elizabethkao/miniforge3/envs/dsc80/lib/python3.10/site-packages/sklearn/metrics/_regression.py:492: FutureWarning:

'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.



In [291]:
fig_pred = px.scatter(
    x=y_test.values[:3000], y=y_pred[:3000],
    opacity=0.3,
    labels={'x': 'Actual Calories', 'y': 'Predicted Calories'},
    title='Final Model: Predicted vs. Actual Calories',
    color_discrete_sequence=['blue']
)
fig_pred.add_shape(type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash'))
fig_pred.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig_pred.show()
fig_pred.write_html('assets/pred_vs_actual.html', include_plotlyjs='cdn')

### Performance Summary

| Model | Test RMSE | Test R² |
|---|---|---|
| Baseline (Linear Regression, 4 features) | 159.16 cal | 0.7721 |
| Final (Linear Regression, 6 features + interaction) | 29.45 cal | 0.9922 |

The final model improved test RMSE from 159.16 to 29.45, which is a reduction of around 130 calories, by adding `carbs`, `sugar`, and a `total_fat x protein` interaction term. Carbohydrates are the single largest caloric source in most recipes and their absence was the main driver of the baseline's high error. Sugar adds resolution within the carb category, and the interaction term captures the extra caloric density of high-fat, high-protein foods that a purely additive model cannot represent.

## Step 8: Fairness Analysis

### Groups

- Group X (High-protein recipes): Recipes in the top quartile of protein PDV (≥ 75th percentile). These are the most "diet-friendly" recipes from a weight-loss perspective and the ones many people on a diet would care about most.
- Group Y (Low-protein recipes): Recipes in the bottom quartile of protein PDV (≤ 25th percentile). These are usually desserts, baked goods, and carbohydrate heavy dishes.



### Fairness Question

Does our model predict calories equally well for high-protein vs. low-protein recipes? 

If the model performs worse on high-protein recipes, the exact recipes a dieter cares most about, that would be a meaningful limitation to discuss.

### Hypotheses & Test

- Null Hypothesis $H_0$: The model is fair. Its RMSE for high-protein and low-protein recipes is roughly the same; any difference is due to random chance.
- Alternative Hypothesis $H_1$: The model is unfair. Its RMSE for the two groups differs.
- Test Statistic: |RMSE(high-protein) − RMSE(low-protein)|
- Significance Level: $\alpha = 0.05$

In [292]:
test_df = X_test_f.copy()
test_df['calories']  = y_test.values
test_df['predicted'] = y_pred
test_df['sq_error']  = (test_df['calories'] - test_df['predicted']) ** 2

q75_p = X_train['protein'].quantile(0.75)
q25_p = X_train['protein'].quantile(0.25)

high_mask = test_df['protein'] >= q75_p
low_mask  = test_df['protein'] <= q25_p

rmse_high = np.sqrt(test_df.loc[high_mask, 'sq_error'].mean())
rmse_low  = np.sqrt(test_df.loc[low_mask,  'sq_error'].mean())
obs_rmse_diff = abs(rmse_high - rmse_low)

print(f"RMSE (high-protein): {rmse_high:.2f}")
print(f"RMSE (low-protein): {rmse_low:.2f}")
print(f"Observed |diff|: {obs_rmse_diff:.2f}")

np.random.seed(42)
sub = test_df[high_mask | low_mask].copy()
n_high_f = high_mask.sum()

perm_diffs_fair = []
for _ in range(10_000):
    idx = np.random.permutation(len(sub))
    r_h = np.sqrt(sub.iloc[idx[:n_high_f]]['sq_error'].mean())
    r_l = np.sqrt(sub.iloc[idx[n_high_f:]]['sq_error'].mean())
    perm_diffs_fair.append(abs(r_h - r_l))

p_val_fair = np.mean(np.array(perm_diffs_fair) >= obs_rmse_diff)
print(f"\nP-value: {p_val_fair:.4f}")
print("Conclusion:", "Reject H₀: evidence of unfairness" if p_val_fair < 0.05
      else "Fail to reject H₀: no significant RMSE difference")

fig_fair = px.histogram(
    x=perm_diffs_fair,
    nbins=50,
    title='Fairness Permutation Test: |RMSE Diff| High vs. Low Protein',
    labels={'x': '|RMSE(high-protein) minus RMSE(low-protein)|'},
    color_discrete_sequence=['lightblue']
)
fig_fair.add_vline(x=obs_rmse_diff, line_color='red', line_dash='dash',
                   annotation_text=f'Observed: {obs_rmse_diff:.1f}')
fig_fair.update_layout(
    yaxis_title='Count',
    plot_bgcolor='white', paper_bgcolor='white'
)
fig_fair.show()
fig_fair.write_html('assets/fairness_test.html', include_plotlyjs='cdn')


RMSE (high-protein): 28.65
RMSE (low-protein): 44.35
Observed |diff|: 15.70

P-value: 0.0008
Conclusion: Reject H₀: evidence of unfairness


### Conclusion

The p value of **0.0008** means we reject the null hypothesis at $\alpha = 0.05$. The model predicts calories significantly less accurately for low-protein recipes, $\text{RMSE} = 44.35$, than for high-protein recipes $\text{RMSE} = 28.65$. This most likely tells us that low-protein recipes like desserts, baked goods, and carb-heavy dishes have more variable caloric density that the macro features don't fully capture.